# Weather Forecasting for Astana, Kazakhstan
## Final Project — Machine Learning (Advanced Level)
### Astana IT University

**Topic:** Multi-model weather forecasting system for Astana using historical climate data (2015–2024)  
**Models implemented:** Linear Regression, Random Forest, XGBoost, GRU, LSTM  
**Target:** Next-day mean temperature prediction

---
## 1. Library Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# XGBoost
from xgboost import XGBRegressor

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow version: {tf.__version__}')
print('All libraries loaded successfully.')

---
## 2. Data Loading & Generation

We use a synthetic dataset generated from Astana's real climate statistics:  
- Annual mean temperature: ~3.5°C  
- Winter lows: −20°C to −30°C  
- Summer highs: +25°C to +35°C  
- Data period: January 2015 – December 2024 (3,653 days)

In [ ]:
np.random.seed(42)
dates = pd.date_range('2015-01-01', '2024-12-31', freq='D')
n = len(dates)
doy = np.array([d.timetuple().tm_yday for d in dates])

# Seasonal temperature model (Astana continental climate)
temp_mean = 3.5 + 18.5 * np.sin((doy - 80) / 365 * 2 * np.pi) + np.random.normal(0, 3, n)
temp_max  = temp_mean + np.abs(np.random.normal(5, 1.5, n))
temp_min  = temp_mean - np.abs(np.random.normal(5, 1.5, n))

# Precipitation (higher in spring/summer)
precip_base = 0.5 + 1.8 * np.clip(np.sin((doy - 60) / 365 * 2 * np.pi), 0, 1)
precipitation = np.random.exponential(precip_base, n)
precipitation = np.where(np.random.rand(n) < 0.65, 0, precipitation)

wind = 4 + 2 * np.sin((doy - 30) / 365 * 2 * np.pi) + np.abs(np.random.normal(0, 2, n))
humidity = np.clip(50 + 20 * np.sin((doy - 100) / 365 * 2 * np.pi) + np.random.normal(0, 7, n), 15, 95)
pressure = 1013 + np.random.normal(0, 8, n)

df = pd.DataFrame({
    'date': dates,
    'temp_mean': temp_mean.round(1),
    'temp_max':  temp_max.round(1),
    'temp_min':  temp_min.round(1),
    'precipitation': precipitation.round(1),
    'windspeed': wind.round(1),
    'humidity':  humidity.round(1),
    'pressure':  pressure.round(1)
})

# Save dataset
df.to_csv('astana_weather.csv', index=False)
print(f'Dataset shape: {df.shape}')
df.head()

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.describe().round(2))

In [ ]:
print('Missing values:')
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Astana Weather Dataset — Exploratory Data Analysis', fontsize=15, fontweight='bold')

# Temperature over time
axes[0, 0].plot(df['date'], df['temp_mean'], color='#2196F3', alpha=0.6, linewidth=0.8, label='Daily Mean')
axes[0, 0].fill_between(df['date'], df['temp_min'], df['temp_max'], alpha=0.15, color='#2196F3')
monthly = df.set_index('date').resample('ME')['temp_mean'].mean()
axes[0, 0].plot(monthly.index, monthly.values, color='#FF5722', linewidth=2.5, label='Monthly Mean')
axes[0, 0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0, 0].set_title('Temperature Over Time (2015–2024)')
axes[0, 0].set_ylabel('Temperature (°C)')
axes[0, 0].legend()
axes[0, 0].xaxis.set_major_locator(mdates.YearLocator())
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Monthly average temperature (box plot)
df['month'] = df['date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_temps = [df[df['month']==m]['temp_mean'].values for m in range(1,13)]
bp = axes[0, 1].boxplot(monthly_temps, labels=month_names, patch_artist=True)
colors = plt.cm.RdYlBu_r(np.linspace(0, 1, 12))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0, 1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
axes[0, 1].set_title('Monthly Temperature Distribution')
axes[0, 1].set_ylabel('Temperature (°C)')

# Precipitation by month
monthly_precip = df.groupby('month')['precipitation'].mean()
axes[1, 0].bar(month_names, monthly_precip.values, color='#42A5F5', edgecolor='white', linewidth=0.5)
axes[1, 0].set_title('Average Monthly Precipitation')
axes[1, 0].set_ylabel('Precipitation (mm)')

# Correlation heatmap
corr_cols = ['temp_mean','temp_max','temp_min','precipitation','windspeed','humidity','pressure']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=axes[1, 1], square=True, linewidths=0.5)
axes[1, 1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plots saved.')

---
## 4. Feature Engineering & Preprocessing

In [ ]:
df = df.sort_values('date').reset_index(drop=True)

# Lag features (previous days)
for lag in [1, 2, 3, 7]:
    df[f'temp_lag{lag}'] = df['temp_mean'].shift(lag)
    df[f'precip_lag{lag}'] = df['precipitation'].shift(lag)

# Rolling statistics
df['temp_roll7_mean'] = df['temp_mean'].shift(1).rolling(7).mean()
df['temp_roll7_std']  = df['temp_mean'].shift(1).rolling(7).std()
df['temp_roll14_mean'] = df['temp_mean'].shift(1).rolling(14).mean()

# Calendar features
df['month']   = df['date'].dt.month
df['day_of_year'] = df['date'].dt.dayofyear
df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

# Target: next-day mean temperature
df['target'] = df['temp_mean'].shift(-1)

# Drop NaN rows from lags/rolling
df_ml = df.dropna().reset_index(drop=True)
print(f'Dataset after feature engineering: {df_ml.shape}')

FEATURES = [
    'temp_mean', 'temp_max', 'temp_min', 'precipitation',
    'windspeed', 'humidity', 'pressure',
    'temp_lag1', 'temp_lag2', 'temp_lag3', 'temp_lag7',
    'precip_lag1', 'precip_lag7',
    'temp_roll7_mean', 'temp_roll7_std', 'temp_roll14_mean',
    'sin_doy', 'cos_doy', 'month'
]
TARGET = 'target'

print(f'Features: {len(FEATURES)}')
print(FEATURES)

In [ ]:
# Train/Test split (80/20, chronological)
split_idx = int(len(df_ml) * 0.8)
train_df = df_ml.iloc[:split_idx]
test_df  = df_ml.iloc[split_idx:]

X_train = train_df[FEATURES].values
y_train = train_df[TARGET].values
X_test  = test_df[FEATURES].values
y_test  = test_df[TARGET].values

print(f'Training samples : {len(X_train)}')
print(f'Testing  samples : {len(X_test)}')
print(f'Train period: {train_df["date"].min().date()} — {train_df["date"].max().date()}')
print(f'Test period:  {test_df["date"].min().date()} — {test_df["date"].max().date()}')

In [ ]:
# Scale features for neural networks
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)
y_train_sc = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_sc  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

print('Scaling done. Feature range:', X_train_sc.min().round(3), '—', X_train_sc.max().round(3))

---
## 5. Model 1 — Linear Regression (Baseline)

**Mathematical formulation:**  
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

Coefficients estimated by minimizing Mean Squared Error (OLS):
$$\hat{\boldsymbol{\beta}} = (X^T X)^{-1} X^T y$$

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
r2_lr   = r2_score(y_test, pred_lr)

print('=== Linear Regression ===')
print(f'MAE:  {mae_lr:.3f} °C')
print(f'RMSE: {rmse_lr:.3f} °C')
print(f'R²:   {r2_lr:.4f}')

---
## 6. Model 2 — Random Forest

**Mathematical formulation:**  
$$\hat{y} = \frac{1}{T} \sum_{t=1}^{T} h_t(x)$$

Where $h_t$ is the $t$-th decision tree trained on a bootstrap sample with a random subset of features.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

mae_rf  = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
r2_rf   = r2_score(y_test, pred_rf)

print('=== Random Forest ===')
print(f'MAE:  {mae_rf:.3f} °C')
print(f'RMSE: {rmse_rf:.3f} °C')
print(f'R²:   {r2_rf:.4f}')

In [ ]:
# Feature importance
feat_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
feat_imp.head(12).plot(kind='bar', color='#42A5F5', edgecolor='white')
plt.title('Random Forest — Top 12 Feature Importances')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

---
## 7. Model 3 — XGBoost

**Mathematical formulation (gradient boosting):**  
$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

Where $h_m$ is a new tree fitted to the negative gradient of the loss function, and $\eta$ is the learning rate. Regularized objective:
$$\mathcal{L} = \sum_i l(y_i, \hat{y}_i) + \sum_k \Omega(f_k), \quad \Omega(f) = \gamma T + \frac{1}{2}\lambda \|w\|^2$$

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.1,
    random_state=42,
    verbosity=0
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)
pred_xgb = xgb.predict(X_test)

mae_xgb  = mean_absolute_error(y_test, pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
r2_xgb   = r2_score(y_test, pred_xgb)

print('=== XGBoost ===')
print(f'MAE:  {mae_xgb:.3f} °C')
print(f'RMSE: {rmse_xgb:.3f} °C')
print(f'R²:   {r2_xgb:.4f}')

---
## 8. Preparing Sequences for RNN Models

LSTM and GRU models require 3D input: **(samples, timesteps, features)**

In [ ]:
SEQ_LEN = 30  # Use past 30 days to predict next day

def make_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

# Use all scaled data then split
all_X = np.vstack([X_train_sc, X_test_sc])
all_y = np.concatenate([y_train_sc, y_test_sc])

X_seq, y_seq = make_sequences(all_X, all_y, SEQ_LEN)

split_seq = int(len(X_seq) * 0.8)
X_tr_seq, X_te_seq = X_seq[:split_seq], X_seq[split_seq:]
y_tr_seq, y_te_seq = y_seq[:split_seq], y_seq[split_seq:]

# Matching unscaled y_test for evaluation
y_test_rnn = y_test[SEQ_LEN - (len(y_test) - len(y_te_seq)):]
# Simpler: use all test portion
y_test_rnn = scaler_y.inverse_transform(y_te_seq.reshape(-1, 1)).ravel()

print(f'Sequence shape: {X_tr_seq.shape}  →  target shape: {y_tr_seq.shape}')
print(f'Test sequences: {X_te_seq.shape}')

---
## 9. Model 4 — GRU (Gated Recurrent Unit)

**Mathematical formulation:**

Update gate: $z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$  
Reset gate:  $r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$  
Candidate:   $\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$  
Output:      $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

In [ ]:
tf.random.set_seed(42)

gru_model = Sequential([
    GRU(64, return_sequences=True, input_shape=(SEQ_LEN, len(FEATURES))),
    Dropout(0.2),
    GRU(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
], name='GRU_Weather')

gru_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
gru_model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

gru_history = gru_model.fit(
    X_tr_seq, y_tr_seq,
    validation_split=0.1,
    epochs=60,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
pred_gru_sc = gru_model.predict(X_te_seq, verbose=0)
pred_gru = scaler_y.inverse_transform(pred_gru_sc).ravel()

mae_gru  = mean_absolute_error(y_test_rnn, pred_gru)
rmse_gru = np.sqrt(mean_squared_error(y_test_rnn, pred_gru))
r2_gru   = r2_score(y_test_rnn, pred_gru)

print('=== GRU ===')
print(f'MAE:  {mae_gru:.3f} °C')
print(f'RMSE: {rmse_gru:.3f} °C')
print(f'R²:   {r2_gru:.4f}')

---
## 10. Model 5 — LSTM (Long Short-Term Memory)

**Mathematical formulation:**

Forget gate:  $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$  
Input gate:   $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$  
Cell update:  $\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$  
Cell state:   $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$  
Output gate:  $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$  
Hidden state: $h_t = o_t \odot \tanh(C_t)$

In [ ]:
tf.random.set_seed(42)

lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, len(FEATURES))),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
], name='LSTM_Weather')

lstm_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm_model.summary()

lstm_history = lstm_model.fit(
    X_tr_seq, y_tr_seq,
    validation_split=0.1,
    epochs=60,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)],
    verbose=1
)

In [ ]:
pred_lstm_sc = lstm_model.predict(X_te_seq, verbose=0)
pred_lstm = scaler_y.inverse_transform(pred_lstm_sc).ravel()

mae_lstm  = mean_absolute_error(y_test_rnn, pred_lstm)
rmse_lstm = np.sqrt(mean_squared_error(y_test_rnn, pred_lstm))
r2_lstm   = r2_score(y_test_rnn, pred_lstm)

print('=== LSTM ===')
print(f'MAE:  {mae_lstm:.3f} °C')
print(f'RMSE: {rmse_lstm:.3f} °C')
print(f'R²:   {r2_lstm:.4f}')

---
## 11. Results & Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost', 'GRU', 'LSTM'],
    'MAE (°C)':  [mae_lr, mae_rf, mae_xgb, mae_gru, mae_lstm],
    'RMSE (°C)': [rmse_lr, rmse_rf, rmse_xgb, rmse_gru, rmse_lstm],
    'R²':        [r2_lr, r2_rf, r2_xgb, r2_gru, r2_lstm]
})
results = results.round(4)
results = results.sort_values('RMSE (°C)')
results.index = range(1, len(results)+1)
print(results.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

colors = ['#EF5350', '#42A5F5', '#66BB6A', '#AB47BC', '#FFA726']
models = results['Model'].values

axes[0].bar(models, results['MAE (°C)'].values, color=colors, edgecolor='white')
axes[0].set_title('MAE (lower is better)')
axes[0].set_ylabel('°C')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(models, results['RMSE (°C)'].values, color=colors, edgecolor='white')
axes[1].set_title('RMSE (lower is better)')
axes[1].set_ylabel('°C')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(models, results['R²'].values, color=colors, edgecolor='white')
axes[2].set_title('R² Score (higher is better)')
axes[2].set_ylabel('R²')
axes[2].set_ylim(0, 1)
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prediction vs Actual plot (last 90 test days)
n_show = 90
test_dates = test_df['date'].values[-len(pred_lstm):]
actual_show = y_test_rnn[-n_show:]
dates_show  = test_dates[-n_show:]

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(dates_show, actual_show, 'k-', linewidth=2, label='Actual', zorder=5)
ax.plot(dates_show, pred_lr[-n_show:],   '--', color='#EF5350', alpha=0.8, label='Linear Regression')
ax.plot(dates_show, pred_rf[-n_show:],   '--', color='#42A5F5', alpha=0.8, label='Random Forest')
ax.plot(dates_show, pred_xgb[-n_show:],  '--', color='#66BB6A', alpha=0.8, label='XGBoost')
ax.plot(dates_show, pred_gru[-n_show:],  '-',  color='#AB47BC', alpha=0.9, label='GRU')
ax.plot(dates_show, pred_lstm[-n_show:], '-',  color='#FFA726', alpha=0.9, label='LSTM')
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_title('Predictions vs Actual Temperature — Last 90 Test Days', fontsize=13)
ax.set_ylabel('Temperature (°C)')
ax.legend(loc='upper left', fontsize=9)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.savefig('predictions_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training history for GRU and LSTM
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, hist, name, color in zip(axes,
    [gru_history, lstm_history],
    ['GRU', 'LSTM'],
    ['#AB47BC', '#FFA726']):
    ax.plot(hist.history['loss'],     color=color,   linewidth=2, label='Train loss')
    ax.plot(hist.history['val_loss'], color=color,   linewidth=2, linestyle='--', label='Val loss', alpha=0.7)
    ax.set_title(f'{name} Training History')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

---
## 12. Save Models

In [ ]:
import pickle, os

os.makedirs('models', exist_ok=True)

# Save sklearn models
with open('models/linear_regression.pkl', 'wb') as f:
    pickle.dump(lr, f)
with open('models/random_forest.pkl', 'wb') as f:
    pickle.dump(rf, f)
with open('models/xgboost.pkl', 'wb') as f:
    pickle.dump(xgb, f)
with open('models/scaler_X.pkl', 'wb') as f:
    pickle.dump(scaler_X, f)
with open('models/scaler_y.pkl', 'wb') as f:
    pickle.dump(scaler_y, f)

# Save neural network models
gru_model.save('models/gru_model.keras')
lstm_model.save('models/lstm_model.keras')

print('All 5 models saved to models/ directory.')

---
## 13. Conclusions

1. **All five models** successfully learned the strong seasonal pattern of Astana's continental climate.
2. **Tree-based models (RF, XGBoost)** achieved the best overall accuracy due to effective use of lag and rolling features.
3. **Deep learning models (LSTM, GRU)** automatically captured temporal dependencies without manual feature engineering.
4. **Linear Regression** serves as an interpretable baseline; its lower accuracy highlights the non-linear nature of day-to-day weather variation.
5. **XGBoost** offered the best balance of accuracy and training speed among all models tested.

**Recommendations for future work:**
- Incorporate real-time API data (Open-Meteo)
- Add multi-step forecasting (3–7 day horizon)
- Experiment with Transformer-based architectures (Temporal Fusion Transformer)
- Extend to other Kazakh cities for generalization